# Lesson 06 · Exercise: a model selection report

**Deadline:** next class. **Individual.** **Deliverable:** this notebook, executed, with every `YOUR TURN` cell completed and every *Decision* paragraph written.
Be ready to talk for 3 minutes about the finding that surprised you most.

## The situation

You are the engineer who has to pick a language model for a real product in **your own domain** (choose one: sentiment of product
reviews in Portuguese, intent of customer-support messages, category of maintenance tickets, or anything you care about).

You will compare three models using the **seven stops** as instruments. No training, no RAG, no fine-tuning, just reading
models and running forward passes, exactly like in the lecture notebook.

| model | year | why it is here |
|---|---|---|
| `gpt2` | 2019 | the model you dissected in class, 124M |
| `HuggingFaceTB/SmolLM2-360M` | 2024 | a modern small model, same drawing, 360M |
| `Qwen/Qwen2.5-0.5B` | 2024 | a modern multilingual model, 494M |

All three are ungated (no login) and run on the free Colab CPU. We use the **base** versions on purpose: same task as GPT-2,
predict the next piece, so the comparison is fair. The instruct versions are an optional extra at the end.

## The five questions

| # | stop | engineering question |
|---|---|---|
| 1 | tokens + positions | what does my text cost, and does it fit? |
| 2 | block + model | what changed in the drawing between 2019 and 2024? |
| 3 | attention | do the modern models resolve references better? |
| 4 | sampling (logits) | **the real application**: zero-shot classification by reading the next-token distribution |
| 5 | sampling (generation) | which one would I ship, at what temperature, and at what speed? |

In [ ]:
# Setup (run once)
!pip -q install "transformers>=4.45" torch matplotlib numpy pandas

import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, time, math
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(0)

MODELS = ["gpt2", "HuggingFaceTB/SmolLM2-360M", "Qwen/Qwen2.5-0.5B"]
SHORT  = {"gpt2": "GPT-2", "HuggingFaceTB/SmolLM2-360M": "SmolLM2", "Qwen/Qwen2.5-0.5B": "Qwen2.5"}

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, attn_implementation="eager")   # float32 on CPU by default
    mdl.eval()
    return tok, mdl

loaded = {name: load(name) for name in MODELS}     # ~2 GB of downloads the first time, then cached
for name, (tok, mdl) in loaded.items():
    print(f"{SHORT[name]:8s} {sum(p.numel() for p in mdl.parameters()):>14,} parameters")

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, time, math
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(0)

MODELS = ["gpt2", "HuggingFaceTB/SmolLM2-360M", "Qwen/Qwen2.5-0.5B"]
SHORT  = {"gpt2": "GPT-2", "HuggingFaceTB/SmolLM2-360M": "SmolLM2", "Qwen/Qwen2.5-0.5B": "Qwen2.5"}

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, attn_implementation="eager")   # float32 on CPU by default
    mdl.eval()
    return tok, mdl

loaded = {name: load(name) for name in MODELS}     # ~2 GB of downloads the first time, then cached
for name, (tok, mdl) in loaded.items():
    print(f"{SHORT[name]:8s} {sum(p.numel() for p in mdl.parameters()):>14,} parameters")

def n_tokens(tok, text): return len(tok.encode(text))

def describe(mdl):
    "The 'counted by hand' row for any Hugging Face causal LM."
    c = mdl.config
    get = lambda *keys: next((getattr(c, k) for k in keys if hasattr(c, k)), None)
    return {
        "blocks":        get("n_layer", "num_hidden_layers"),
        "width d":       get("n_embd", "hidden_size"),
        "heads":         get("n_head", "num_attention_heads"),
        "kv heads":      get("num_key_value_heads") or get("n_head", "num_attention_heads"),
        "FFN width":     get("n_inner", "intermediate_size") or 4 * get("n_embd", "hidden_size"),
        "context":       get("n_positions", "max_position_embeddings"),
        "vocab":         get("vocab_size"),
        "positions":     "RoPE (rotations inside attention)" if hasattr(c, "rope_theta") else "learned table",
        "tied readout":  bool(getattr(c, "tie_word_embeddings", True)),
        "parameters":    sum(p.numel() for p in mdl.parameters()),
    }

def params_by_component(mdl):
    "Group parameters the way the lecture counted them."
    groups = {"embeddings": 0, "positions": 0, "blocks": 0, "readout (untied)": 0, "other": 0}
    for name, p in mdl.named_parameters():
        n = p.numel()
        if "wpe" in name:                                   groups["positions"] += n
        elif "wte" in name or "embed_tokens" in name:       groups["embeddings"] += n
        elif ".h." in name or ".layers." in name:           groups["blocks"] += n
        elif "lm_head" in name:                             groups["readout (untied)"] += n
        else:                                               groups["other"] += n
    return groups

def attention_maps(tok, mdl, sentence):
    "Returns (labels, attentions) where attentions[L][h] is a (T, T) numpy table."
    ids = tok(sentence, return_tensors="pt").input_ids
    with torch.no_grad():
        out = mdl(ids, output_attentions=True)
    labels = [t.replace("Ġ", "␣").replace("▁", "␣").replace("Ċ", "⏎") for t in tok.convert_ids_to_tokens(ids[0])]
    return labels, [a[0].to(torch.float32).numpy() for a in out.attentions]

def heatmap(matrix, labels, title="", ax=None):
    ax = ax or plt.gca()
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7); ax.set_title(title, fontsize=9)

def label_probs(tok, mdl, prompt, labels):
    "Zero-shot classification by reading the distribution: P(first piece of each label | prompt)."
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = mdl(ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    out = {}
    for lab in labels:
        first = tok.encode(lab, add_special_tokens=False)[0]        # the first piece of the label
        out[lab] = float(probs[first])
    return out

def tokens_per_second(tok, mdl, prompt, n_new=30):
    ids = tok(prompt, return_tensors="pt").input_ids
    t0 = time.perf_counter()
    with torch.no_grad():
        mdl.generate(ids, max_new_tokens=n_new, do_sample=False, pad_token_id=tok.eos_token_id or tok.pad_token_id)
    return n_new / (time.perf_counter() - t0)

print("helpers ready")

## Q1 · Cost and context (stops 1 and 3)

Take a **real** text from your domain, at least 200 words, in Portuguese, plus its English version (translate it with any tool, or use a bilingual document).
Tokenize it with the three tokenizers.

In [ ]:
# YOUR TURN · paste your own texts (≥ 200 words each). These two are placeholders, replace them.
text_pt = """Substitua este texto por um texto real do seu domínio, com pelo menos duzentas palavras."""
text_en = """Replace this with the English version of your text, at least two hundred words."""

rows = []
for name, (tok, mdl) in loaded.items():
    rows.append({"model": SHORT[name],
                 "vocab": len(tok),
                 "context": describe(mdl)["context"],
                 "PT tokens": n_tokens(tok, text_pt), "PT tokens/word": n_tokens(tok, text_pt) / len(text_pt.split()),
                 "EN tokens": n_tokens(tok, text_en), "EN tokens/word": n_tokens(tok, text_en) / len(text_en.split())})
q1 = pd.DataFrame(rows).set_index("model").round(2)
display(q1)

q1[["PT tokens/word", "EN tokens/word"]].plot.bar(figsize=(6, 3), color=["#6E9277", "#1E3261"], title="tokens per word")
plt.ylabel("tokens/word"); plt.xticks(rotation=0); plt.show()

# How many copies of your PT text fit in each context window?
for name, (tok, mdl) in loaded.items():
    print(f"{SHORT[name]:8s} context {describe(mdl)['context']:>6,} → fits {describe(mdl)['context'] // n_tokens(tok, text_pt)} copies of your text")

**Decision 1** (write 4 to 6 lines). At a hypothetical price per token, how much more does Portuguese cost on each model, and what does the context window mean for a document of your domain? Which tokenizer would you want for a Portuguese product, and why?

> *your answer here*

## Q2 · Anatomy (stops 5 and 7)

Build the "counted by hand" table for the three models, read straight from the configuration and the parameters.

In [ ]:
anatomy = pd.DataFrame({SHORT[n]: describe(m) for n, (t, m) in loaded.items()})
display(anatomy)

comp = pd.DataFrame({SHORT[n]: params_by_component(m) for n, (t, m) in loaded.items()})
display(comp)
(comp / comp.sum()).T.plot.bar(stacked=True, figsize=(7, 3), title="where the parameters live (share)")
plt.xticks(rotation=0); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.show()

In [ ]:
# YOUR TURN · print one block of each model and identify, in the printed code, the four pieces of the lecture:
# norm, attention, residual (hint: it is not printed, why?), FFN. Note the names each library uses for them.
for name, (tok, mdl) in loaded.items():
    print("=" * 30, SHORT[name], "=" * 30)
    blocks = getattr(getattr(mdl, "transformer", None), "h", None) or mdl.model.layers
    print(blocks[0])

**Decision 2** (4 to 6 lines). What changed in the drawing between 2019 and 2024? Consider: positions (learned table vs RoPE),
the ratio of blocks to width, the number of key/value heads vs query heads, the share of parameters in the embedding table, tied readout. Which changes are about *quality* and which are about *cost*?

> *your answer here*

## Q3 · Attention (stop 4)

Write **two sentences from your domain with an ambiguous reference** (a pronoun that could point to two nouns), one in Portuguese and one in English.
For each model, find the head where the pronoun attends most to the correct noun, and show it.

In [ ]:
# YOUR TURN · replace the sentences and the three words. The pronoun and both candidate nouns must each be one piece
# in every tokenizer (check the printed labels; pick different words if a noun gets split).
sentence = "The animal didn't cross the street because it was too tired"
pronoun, correct, wrong = "it", "animal", "street"

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, (tok, mdl)) in zip(axes, loaded.items()):
    labels, att = attention_maps(tok, mdl, sentence)
    find = lambda w: next(i for i, l in enumerate(labels) if l.strip("␣") == w)
    p, c, w = find(pronoun), find(correct), find(wrong)
    best = max(((L, h, att[L][h][p, c], att[L][h][p, w]) for L in range(len(att)) for h in range(att[0].shape[0])),
               key=lambda r: r[2])
    L, h, wc, ww = best
    heatmap(att[L][h], labels, f"{SHORT[name]} · layer {L} head {h}\n{pronoun}→{correct} {wc:.2f} · {pronoun}→{wrong} {ww:.2f}", ax)
plt.tight_layout(); plt.show()

In [ ]:
# YOUR TURN · now the Portuguese sentence. Copy the cell above and adapt it.

**Decision 3** (4 to 6 lines). Did the modern models resolve the reference more clearly (higher weight on the right noun, lower on the wrong one)? Was Portuguese different from English? Remember: heads' roles emerge, and a single head is weak evidence. What would you need to claim this properly?

> *your answer here*

## Q4 · The real application: zero-shot classification by reading the distribution (stop 6)

No generation. You write a prompt that ends right before the label, and you read the probability of each label's first piece
at the next position, exactly the softmax of stop 6. Whichever label is more probable is the prediction.

Build a labelled set from your domain, **at least 30 examples**. Three starter sets are below (10 each). Extend the one closest to
your domain, or replace it with your own.

In [ ]:
STARTERS = {
  "sentiment_pt": {
    "template": "Avaliação: {text}\nSentimento (positivo ou negativo):",
    "labels": [" positivo", " negativo"],
    "data": [
      ("Produto excelente, chegou antes do prazo e funciona perfeitamente.", " positivo"),
      ("Veio com defeito e o suporte nunca respondeu.", " negativo"),
      ("Bateria dura o dia inteiro, recomendo.", " positivo"),
      ("Parou de funcionar depois de uma semana.", " negativo"),
      ("Ótimo custo benefício, comprarei de novo.", " positivo"),
      ("Embalagem amassada e o item riscado.", " negativo"),
      ("Instalação simples e resultado acima do esperado.", " positivo"),
      ("Não corresponde à descrição do anúncio.", " negativo"),
      ("Atendimento rápido e cordial.", " positivo"),
      ("Cobraram duas vezes e não devolveram o valor.", " negativo"),
    ]},
  "support_intent": {
    "template": "Customer message: {text}\nIntent (refund, shipping, or technical):",
    "labels": [" refund", " shipping", " technical"],
    "data": [
      ("I want my money back, the product is broken.", " refund"),
      ("Where is my order? It has been two weeks.", " shipping"),
      ("The app crashes every time I open the settings.", " technical"),
      ("Can I return this and get a refund?", " refund"),
      ("The tracking number does not work.", " shipping"),
      ("Login fails with error 403.", " technical"),
      ("Charged twice, please reverse one payment.", " refund"),
      ("Package arrived at the wrong address.", " shipping"),
      ("Bluetooth will not pair with my phone.", " technical"),
      ("Do you ship to Brazil?", " shipping"),
    ]},
  "ticket_category": {
    "template": "Maintenance ticket: {text}\nCategory (electrical, mechanical, or software):",
    "labels": [" electrical", " mechanical", " software"],
    "data": [
      ("Motor bearing makes a grinding noise at high speed.", " mechanical"),
      ("PLC firmware update failed and the line stopped.", " software"),
      ("Circuit breaker trips when the conveyor starts.", " electrical"),
      ("Hydraulic press leaking oil from the main seal.", " mechanical"),
      ("HMI screen frozen, needs a reboot every hour.", " software"),
      ("Cable insulation melted near the junction box.", " electrical"),
      ("Belt tension out of spec, slipping under load.", " mechanical"),
      ("SCADA alarms not logging since the last patch.", " software"),
      ("Voltage drop on phase B under load.", " electrical"),
      ("Gearbox vibration above threshold.", " mechanical"),
    ]},
}

# YOUR TURN · pick a starter and extend it to ≥ 30 examples, or build your own dict with the same shape.
task = STARTERS["sentiment_pt"]
print(len(task["data"]), "examples ·", task["labels"])

In [ ]:
def evaluate(tok, mdl, task, few_shot=0):
    "Accuracy of zero-shot (or few-shot) classification by label probability."
    demos = ""
    if few_shot:
        demos = "\n\n".join(task["template"].format(text=t) + lab for t, lab in task["data"][:few_shot]) + "\n\n"
    data = task["data"][few_shot:]
    correct = 0
    for text, gold in data:
        probs = label_probs(tok, mdl, demos + task["template"].format(text=text), task["labels"])
        pred = max(probs, key=probs.get)
        correct += (pred == gold)
    return correct / len(data)

rows = []
for name, (tok, mdl) in loaded.items():
    rows.append({"model": SHORT[name],
                 "zero-shot acc": evaluate(tok, mdl, task, few_shot=0),
                 "3-shot acc":    evaluate(tok, mdl, task, few_shot=3)})
q4 = pd.DataFrame(rows).set_index("model").round(3)
display(q4)
q4.plot.bar(figsize=(6, 3), color=["#1E3261", "#6E9277"], title="classification by reading the next-token distribution")
plt.axhline(1 / len(task["labels"]), color="gray", ls="--", lw=1, label="chance"); plt.legend(); plt.xticks(rotation=0); plt.show()

In [ ]:
# Look inside one prediction: the whole top-10 distribution, not just the labels. Do the labels even appear?
name = MODELS[2]; tok, mdl = loaded[name]
text, gold = task["data"][0]
prompt = task["template"].format(text=text)
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    probs = torch.softmax(mdl(ids).logits[0, -1], dim=-1)
top = torch.topk(probs, 10)
print(f"{SHORT[name]} · next piece after the prompt · gold = {gold!r}")
for v, i in zip(top.values, top.indices):
    print(f"  {v:.3f}  {tok.decode([int(i)])!r}")

**Decision 4** (6 to 8 lines). Which model would you use for this classification, and is zero-shot good enough for a product? Discuss: the effect of few-shot, the cases each model gets wrong (print some), and one limitation of reading only the *first piece* of each label (hint: check how each tokenizer splits your labels).

> *your answer here*

## Q5 · Generation and the cost of inference (stop 6)

One prompt from your domain, three temperatures, three models. Then measure speed.

In [ ]:
# YOUR TURN · a prompt from your domain that asks for a factual continuation (dates, numbers, names invite hallucination).
prompt = "The maintenance manual for the hydraulic press, section 3, states that the oil must be replaced every"

for name, (tok, mdl) in loaded.items():
    print("=" * 20, SHORT[name], "=" * 20)
    ids = tok(prompt, return_tensors="pt").input_ids
    for T in [0.2, 0.7, 1.5]:
        with torch.no_grad():
            out = mdl.generate(ids, max_new_tokens=30, do_sample=True, temperature=T,
                               pad_token_id=tok.eos_token_id or tok.pad_token_id)
        print(f"T={T}: ...{tok.decode(out[0][ids.shape[1]:])!r}")

In [ ]:
rows = []
for name, (tok, mdl) in loaded.items():
    size_mb = sum(p.numel() for p in mdl.parameters()) * 4 / 1e6      # float32
    rows.append({"model": SHORT[name], "size (MB, fp32)": round(size_mb), "tokens/s (this machine)": round(tokens_per_second(tok, mdl, prompt), 1)})
display(pd.DataFrame(rows).set_index("model"))

**Decision 5** (6 to 8 lines). Which model would you ship, at which temperature, and why? Weigh quality (Q3, Q4), cost (Q1, Q5), and risk (the hallucinations you saw). Name one thing you would need that this notebook cannot measure.

> *your answer here*

## Final table and decision

Fill the table with your numbers. One line per model.

In [ ]:
final = pd.DataFrame({
    "model":              [SHORT[n] for n in MODELS],
    "params (M)":         [round(describe(loaded[n][1])["parameters"] / 1e6) for n in MODELS],
    "PT tokens/word":     list(q1["PT tokens/word"]),
    "context":            [describe(loaded[n][1])["context"] for n in MODELS],
    "zero-shot acc":      list(q4["zero-shot acc"]),
    "3-shot acc":         list(q4["3-shot acc"]),
    "tokens/s":           [r["tokens/s (this machine)"] for r in rows],
}).set_index("model")
display(final)

**Final decision** (one paragraph). Which model, for which product, with which settings, and what is the main risk you accept?

> *your answer here*

---

### Optional extras

- Load `Qwen/Qwen2.5-0.5B-Instruct` and repeat Q4 using `tok.apply_chat_template(...)` to build the prompt. Does instruction tuning help zero-shot classification?
- In Q3, average the pronoun→noun weight over *all* heads of a layer instead of picking the best head. Is the conclusion more robust?
- Replace `float32` with `torch.float16` on a GPU runtime and measure tokens/s again.

In [ ]:
## Resources